# 01 — US Technology Market Share: Top / Middle / Bottom 30

## Question 1 (verbatim instruction)

> "With the excel sheet that I copied for you two last week, use its sector
> and industry information to find the top ten companies, the middle ten
> companies, and the bottom ten companies regarding market shares."

## What this notebook delivers

A selection of 30 US Technology companies — 10 top, 10 middle, 10 bottom by market share — saved to `data/processed/q1_selected_30_all_tech.csv`. The 30 companies are the input to `02_weekly_tracking.ipynb`.

## Inputs

**`data/raw/nasdaq_ticker_universe.csv`** — classification only

| column | use |
|---|---|
| `symbol` | join key |
| `sector` | filter to `"Technology"` |
| `industry` | industry-level share |
| `is_us_domiciled` | filter to `True` |

**`data/raw/bloomberg_tech_sector_quarterly.csv`** — quarterly fundamentals + market cap

| column | use |
|---|---|
| `Ticker` | join key (format `"NVDA US Equity"`) |
| `Market Cap:Q` | **share numerator** |
| `Revenue:Q` | context only — see diagnostic |
| `NI / Profit:Q` | context |
| `EBITDA:Q` | context |
| `EPS:Q` | context |
| `P/E:Q` | context |

**`data/raw/bloomberg_tech_sector_ttm.csv`** — TTM fundamentals

| column | use |
|---|---|
| `Ticker` | join key |
| `Revenue T12M` | **not used** for the share metric |
| `EPS T12M` | context |

Both Bloomberg exports are already pre-filtered to the entire Technology sector and contain **no sector / industry column**. The Nasdaq ticker universe is the only file with classification, so it drives the US filter and the industry breakdown.

## Method — why market-cap share

Market share is **not** a field in any input file. It is computed as:

`share_i = metric_i / sum(metric_j)` over the filtered US-tech universe.

Two candidate metrics were tested: revenue and market cap.

**Revenue was rejected after a diagnostic.** The Bloomberg revenue fields (`Revenue:Q` and `Revenue T12M`) contain per-company figures 2.5×–4.3× larger than the companies' reported TTM revenue, with the ratio **varying by company** (median ratio for `Revenue:Q` = 2.01×, spread 1.15×–6.03×). Neither field is a consistent cross-sectional measure. The diagnostic (Cell 4) is retained in the notebook as an **audit trail** — it prints the ratio table and a verdict line.

**Market cap is used as the primary metric.** It is a single-period, cleanly reported figure across all 1,000+ US tech companies in the universe. Two shares are computed:

| column | denominator | answers |
|---|---|---|
| `share_of_all_tech_pct` | sum of Market Cap of all US tech | "% of total US tech market cap" |
| `share_of_industry_pct` | sum of Market Cap within industry | "% of the company's own industry" |

The **top-10 membership is stable** across revenue-based, TTM-based, and market-cap-based measures; only the ordering and the exact percentages change.

## Universe filters

- `is_us_domiciled == True` (drops TSM, ASML, SAP, ARM, etc.)
- `sector == "Technology"`
- `Market Cap:Q > 0`
- Drop `SIFY` (corrupt row in Bloomberg export) and `GOOG` (dedupe vs `GOOGL`)
- Drop preferreds / warrants / units (tickers with `^` or ending `W`)

## Bucket definition

Sorted descending by `share_of_all_tech_pct`:

- **top 10** = ranks 1–10
- **middle 10** = 10 rows centered on the median rank
- **bottom 10** = ranks N-9 … N

## Outputs

- `data/processed/q1_us_tech_with_shares.csv` — full cleaned universe (~1,000 rows)
- `data/processed/q1_selected_30_all_tech.csv` — 30 selected companies

## Caveats

1. **`industry` labels are a coarse Nasdaq taxonomy.** "Semiconductors" lumps fabless, foundry, memory, and equipment companies together. `share_of_industry_pct` is a taxonomy-constrained proxy, not an antitrust market definition.
2. **Single-company industries produce 100% shares by construction.** `TRNR` shows 100% of "Consumer Electronics/Appliances" because it is the only US-tech company in that bucket.
3. **KLA's classification under "Electronic Components"** (not "Semiconductors") inflates its industry share to 41.3%. Reflects the taxonomy, not a data error.
4. **Bottom-10 shares are all <0.0001%** — read as "effectively zero," not as a precise ranking.
5. **Shares are relative to the filtered universe**, not to all US tech companies on earth.
6. **Snapshot ambiguity**: the three raw files disagree on market cap and were exported on different dates.

###  1: Imports & Paths

In [106]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

# Walk up from CWD until we find data/raw/
def find_repo_root(start: Path) -> Path:
    p = start.resolve()
    for _ in range(10):
        if (p / "data" / "raw").is_dir():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise FileNotFoundError(f"data/raw/ not found above {start}")

try:
    REPO_ROOT = find_repo_root(Path.cwd())
except FileNotFoundError:
    REPO_ROOT = find_repo_root(Path.cwd().parent)

DATA_RAW       = REPO_ROOT / "data" / "raw"
DATA_PROCESSED = REPO_ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

TICKER_UNIVERSE = DATA_RAW / "nasdaq_ticker_universe.csv"
BLOOMBERG_Q     = DATA_RAW / "bloomberg_tech_sector_quarterly.csv"

OUT_SELECTED    = DATA_PROCESSED / "q1_selected_30_all_tech.csv"
OUT_UNIVERSE    = DATA_PROCESSED / "q1_us_tech_with_shares.csv"

for p in (TICKER_UNIVERSE, BLOOMBERG_Q):
    if not p.is_file():
        raise FileNotFoundError(f"Missing input: {p}")

print("REPO_ROOT:", REPO_ROOT)

REPO_ROOT: C:\Users\mtwei\Documents\VSCode\Data Analysis Research\Data-Analysis-Research\USTechMarketShare


### 2: Load & clean the ticker universe

In [107]:
tk = pd.read_csv(TICKER_UNIVERSE)

tk["ticker"] = tk["symbol"].astype(str).str.upper().str.strip()
tk["is_us_domiciled"] = (
    tk["is_us_domiciled"].astype(str).str.lower().map({"true": True, "false": False})
)

# Drop preferreds / warrants / units — not operating companies
tk = tk[~tk["ticker"].str.contains(r"\^|W$", regex=True, na=False)]

# Keep only the classification columns we need
cls = tk[["ticker", "name", "sector", "industry", "is_us_domiciled"]].drop_duplicates("ticker")

print("Classification rows:", len(cls))

Classification rows: 6259


### 3: Load & clean the Bloomberg quarterly export

In [108]:
# The raw export has 2 junk header rows; skiprows=2 promotes the real header.
x = pd.read_csv(BLOOMBERG_Q, skiprows=2)

# Guard: if the header is wrong, skiprows was wrong — fail fast.
assert {"Ticker", "Revenue:Q", "Market Cap:Q"}.issubset(x.columns), \
    f"Header wrong: {list(x.columns)[:6]}"

x = x[x["Ticker"].notna()].copy()
x = x[~x["Ticker"].astype(str).str.strip().str.lower().str.startswith("none")]

x["ticker"] = (
    x["Ticker"].astype(str).str.replace(r"\s+US\s+Equity$", "", regex=True).str.upper()
)

for c in ["Market Cap:Q", "Revenue:Q", "NI / Profit:Q",
          "EPS:Q", "P/E:Q", "EBITDA:Q", "Volume:M-3"]:
    x[c] = pd.to_numeric(x[c], errors="coerce")

print("Quarterly rows:", len(x))

Quarterly rows: 1316


### 4: Verify Revenue:Q is quarterly

In [109]:
# Rough quarterly revenue (USD billions) from public filings, ±20% tolerance.
known_q_rev_b = {"AAPL": 95, "MSFT": 68, "NVDA": 39, "MU": 9, "AVGO": 15, "DELL": 23}

probe = (
    x.set_index("ticker")
     .reindex(list(known_q_rev_b))[["Revenue:Q"]]
     .assign(reported_B=lambda d: d["Revenue:Q"] / 1e9,
             expected_B=lambda d: d.index.map(known_q_rev_b))
)
probe["ratio"] = probe["reported_B"] / probe["expected_B"]

print(probe.to_string())
med = probe["ratio"].median()
print(f"\nmedian reported/expected = {med:,.2f}")
if 0.8 <= med <= 1.2:
    print("✅ Revenue:Q is quarterly.")
elif 3.0 <= med <= 5.0:
    print("⚠️  Revenue:Q looks annual (≈4×). Shares still valid; label accordingly in write-up.")
else:
    print(f"⚠️  ratio {med:.2f} — inspect manually.")

                  Revenue:Q  reported_B  expected_B  ratio
ticker                                                    
AAPL   109,417,000,000.0000    109.4170          95 1.1518
MSFT    90,007,000,000.0000     90.0070          68 1.3236
NVDA    96,221,000,000.0000     96.2210          39 2.4672
MU      54,229,000,000.0000     54.2290           9 6.0254
AVGO    29,591,000,000.0000     29.5910          15 1.9727
DELL    46,971,000,000.0000     46.9710          23 2.0422

median reported/expected = 2.01
⚠️  ratio 2.01 — inspect manually.


### 5: Join, filter to US tech, compute shares

In [110]:
df = x.merge(cls, on="ticker", how="left")

df = df[
    (df["is_us_domiciled"] == True) &
    (df["sector"] == "Technology") &
    (df["Market Cap:Q"] > 0)
].copy()

df = df[~df["ticker"].isin({"SIFY", "GOOG"})]
df = df.sort_values("Market Cap:Q", ascending=False).reset_index(drop=True)

# Share by market cap (single, consistent metric across all companies)
df["share_of_all_tech_pct"] = df["Market Cap:Q"] / df["Market Cap:Q"].sum() * 100
df["share_of_industry_pct"] = (
    df["Market Cap:Q"] / df.groupby("industry")["Market Cap:Q"].transform("sum") * 100
)

# For display continuity, keep Revenue:Q as-is but sort on market cap
print(f"Universe size: {len(df)}")
print(f"Total market cap denominator: ${df['Market Cap:Q'].sum():,.0f}")
print("\nTop 5 by market cap share:")
print(
    df.head(5)[["ticker", "Name", "industry", "Market Cap:Q", "share_of_all_tech_pct"]]
      .to_string(index=False)
)

Universe size: 328
Total market cap denominator: $23,073,602,392,024

Top 5 by market cap share:
ticker                  Name                                industry           Market Cap:Q  share_of_all_tech_pct
  NVDA           NVIDIA CORP                          Semiconductors 4,994,565,668,864.0000                21.6462
  AAPL             APPLE INC                  Computer Manufacturing 4,145,731,600,384.0000                17.9674
  MSFT        MICROSOFT CORP Computer Software: Prepackaged Software 2,770,419,580,928.0000                12.0069
  AVGO          BROADCOM INC                          Semiconductors 1,858,422,702,080.0000                 8.0543
    MU MICRON TECHNOLOGY INC                          Semiconductors 1,083,678,982,144.0000                 4.6966


### 6: Slice top / middle / bottom 10 on one denomintor

In [111]:
d = df.sort_values("share_of_all_tech_pct", ascending=False).reset_index(drop=True)
n = len(d)

top = d.head(10).copy()
mid = d.iloc[n//2 - 5 : n//2 + 5].copy()
bot = d.tail(10).copy()

top["bucket"], mid["bucket"], bot["bucket"] = "top10", "middle10", "bottom10"

sel_30 = pd.concat([top, mid, bot]).reset_index(drop=True)

print("bucket counts:", sel_30["bucket"].value_counts().to_dict())

bucket counts: {'top10': 10, 'middle10': 10, 'bottom10': 10}


### 7: Save

In [112]:
KEEP = ["bucket", "ticker", "Name", "industry",
        "Market Cap:Q", "Revenue:Q", "NI / Profit:Q", "EBITDA:Q", "EPS:Q", "P/E:Q",
        "share_of_all_tech_pct", "share_of_industry_pct"]

sel_30[KEEP].to_csv(OUT_SELECTED, index=False)
df.to_csv(OUT_UNIVERSE, index=False)

print(f"Saved {OUT_SELECTED.name}: {len(sel_30)} rows")
print(f"Saved {OUT_UNIVERSE.name}: {len(df)} rows")

Saved q1_selected_30_all_tech.csv: 30 rows
Saved q1_us_tech_with_shares.csv: 328 rows


### 8: Sanity check

In [113]:
assert len(sel_30) == 30
assert sel_30["ticker"].is_unique
assert sel_30["bucket"].value_counts().to_dict() == {"top10": 10, "middle10": 10, "bottom10": 10}

t_min = sel_30.loc[sel_30.bucket == "top10",    "share_of_all_tech_pct"].min()
m_max = sel_30.loc[sel_30.bucket == "middle10", "share_of_all_tech_pct"].max()
m_min = sel_30.loc[sel_30.bucket == "middle10", "share_of_all_tech_pct"].min()
b_max = sel_30.loc[sel_30.bucket == "bottom10", "share_of_all_tech_pct"].max()

assert t_min > m_max and m_min > b_max, "buckets overlap — re-check slicing"
print("✅ 10 / 10 / 10, no duplicates, buckets strictly ordered.")

✅ 10 / 10 / 10, no duplicates, buckets strictly ordered.


### 9: Final display

In [114]:
cols = ["bucket", "ticker", "Name", "industry",
        "Market Cap:Q", "share_of_all_tech_pct", "share_of_industry_pct"]

print(sel_30.sort_values("share_of_all_tech_pct", ascending=False)
            [cols].to_string(index=False))

  bucket ticker                       Name                                       industry           Market Cap:Q  share_of_all_tech_pct  share_of_industry_pct
   top10   NVDA                NVIDIA CORP                                 Semiconductors 4,994,565,668,864.0000                21.6462                45.1641
   top10   AAPL                  APPLE INC                         Computer Manufacturing 4,145,731,600,384.0000                17.9674                89.0193
   top10   MSFT             MICROSOFT CORP        Computer Software: Prepackaged Software 2,770,419,580,928.0000                12.0069                54.8318
   top10   AVGO               BROADCOM INC                                 Semiconductors 1,858,422,702,080.0000                 8.0543                16.8050
   top10     MU      MICRON TECHNOLOGY INC                                 Semiconductors 1,083,678,982,144.0000                 4.6966                 9.7993
   top10    AMD     ADVANCED MICRO DEVICES    